In [1]:
import pandas as pd
import numpy as np
from math import floor
import random
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import binom
from scipy.special import comb
from scipy.stats import ttest_ind

import ast


In [ ]:
dfWLO = pd.read_csv("../../soccer_binary.csv")

dfWLO.head()

,League,Season,Team,Sequence,seq_len
0,Bundesliga,2000,Bayern Munich,"[1, 1, 1, 0, 1, 1, 0, 0, 1, 1, 0, 0, 1, 1, 1, ...",35
1,Bundesliga,2000,Bochum,"[1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, ...",34
2,Bundesliga,2000,Cottbus,"[0, 0, 0, 1, 0, 0, 1, 0, 1, 1, 0, 0, 1, 0, 0, ...",35
3,Bundesliga,2000,Dortmund,"[1, 1, 0, 1, 1, 0, 1, 0, 0, 0, 1, 1, 1, 1, 1, ...",35
4,Bundesliga,2000,Ein Frankfurt,"[1, 0, 1, 0, 1, 0, 0, 1, 0, 1, 1, 0, 0, 0, 0, ...",34


In [3]:
dfWLO['Sequence'] = dfWLO['Sequence'].apply(ast.literal_eval)

In [4]:
dfWLO['League'] = dfWLO['League'].replace("E0", "Premier_League")
dfWLO['League'] = dfWLO['League'].replace("D1", "Bundesliga")
dfWLO['League'] = dfWLO['League'].replace("SP1", "La_Liga")

In [5]:
dfWLO['Total_Games'] = dfWLO['Sequence'].str.len()
dfWLO['Wins'] = dfWLO['Sequence'].apply(sum)
dfWLO

,League,Season,Team,Sequence,seq_len,Total_Games,Wins
0,Bundesliga,2000,Bayern Munich,"[1, 1, 1, 0, 1, 1, 0, 0, 1, 1, 0, 0, 1, 1, 1, ...",35,29,19
1,Bundesliga,2000,Bochum,"[1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, ...",34,28,7
2,Bundesliga,2000,Cottbus,"[0, 0, 0, 1, 0, 0, 1, 0, 1, 1, 0, 0, 1, 0, 0, ...",35,32,13
3,Bundesliga,2000,Dortmund,"[1, 1, 0, 1, 1, 0, 1, 0, 0, 0, 1, 1, 1, 1, 1, ...",35,25,17
4,Bundesliga,2000,Ein Frankfurt,"[1, 0, 1, 0, 1, 0, 0, 1, 0, 1, 1, 0, 0, 0, 0, ...",34,29,10
...,...,...,...,...,...,...,...
1428,La_Liga,2024,Sociedad,"[0, 1, 0, 0, 0, 1, 1, 0, 1, 1, 0, 1, 1, 0, 1, ...",38,31,13
1429,La_Liga,2024,Valencia,"[0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 1, ...",38,25,11
1430,La_Liga,2024,Valladolid,"[1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, ...",38,34,4
1431,La_Liga,2024,Vallecano,"[1, 0, 0, 1, 1, 0, 1, 0, 0, 0, 1, 1, 1, 1, 1, ...",38,25,13


In [6]:
dfWLO['n2'] = dfWLO['Total_Games'] // 2
dfWLO['n1'] = dfWLO['Total_Games'] - dfWLO['n2']
def secondhalf(seq):
    k1 = sum(seq[len(seq)//2:])
    k2 = sum(seq) - k1
    return k2
dfWLO['k2'] = dfWLO['Sequence'].apply(secondhalf)
dfWLO['k1'] = dfWLO['Wins'] - dfWLO['k2']
dfWLO['p1'] = dfWLO['k1'] / dfWLO['n1']
dfWLO['p2'] = dfWLO['k2'] / dfWLO['n2']

In [7]:
def c1(n2,k2, p):
    total = 0
    for i in range(k2+1):
    	total += comb(n2,i) * p**i * (1-p)**(n2-i)
    if(total==0):
    	return math.inf
    else:
    	return 1 / total
dfWLO['p'] = dfWLO['Wins']/dfWLO['Total_Games']

In [8]:
dfWLO['c1'] = dfWLO.apply(lambda row: c1(row['n2'], row['k2'], row['p']), axis=1)
dfWLO

,League,Season,Team,Sequence,seq_len,Total_Games,Wins,n2,n1,k2,k1,p1,p2,p,c1
0,Bundesliga,2000,Bayern Munich,"[1, 1, 1, 0, 1, 1, 0, 0, 1, 1, 0, 0, 1, 1, 1, ...",35,29,19,14,15,9,10,0.666667,0.642857,0.655172,1.782253
1,Bundesliga,2000,Bochum,"[1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, ...",34,28,7,14,14,5,2,0.142857,0.357143,0.250000,1.125706
2,Bundesliga,2000,Cottbus,"[0, 0, 0, 1, 0, 0, 1, 0, 1, 1, 0, 0, 1, 0, 0, ...",35,32,13,16,16,6,7,0.437500,0.375000,0.406250,1.974245
3,Bundesliga,2000,Dortmund,"[1, 1, 0, 1, 1, 0, 1, 0, 0, 0, 1, 1, 1, 1, 1, ...",35,25,17,12,13,7,10,0.769231,0.583333,0.680000,3.023294
4,Bundesliga,2000,Ein Frankfurt,"[1, 0, 1, 0, 1, 0, 0, 1, 0, 1, 1, 0, 0, 0, 0, ...",34,29,10,14,15,6,4,0.266667,0.428571,0.344828,1.208576
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1428,La_Liga,2024,Sociedad,"[0, 1, 0, 0, 0, 1, 1, 0, 1, 1, 0, 1, 1, 0, 1, ...",38,31,13,15,16,8,5,0.312500,0.533333,0.419355,1.141987
1429,La_Liga,2024,Valencia,"[0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 1, ...",38,25,11,12,13,2,9,0.692308,0.166667,0.440000,20.544653
1430,La_Liga,2024,Valladolid,"[1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, ...",38,34,4,17,17,4,0,0.000000,0.235294,0.117647,1.043195
1431,La_Liga,2024,Vallecano,"[1, 0, 0, 1, 1, 0, 1, 0, 0, 0, 1, 1, 1, 1, 1, ...",38,25,13,12,13,6,7,0.538462,0.500000,0.520000,1.793021


In [9]:
def c2(pq, p2):
    pval = ttest_ind(p1, p2, alternative = 'two-sided')
    if (pval == 0):
        return math.inf
    else:
        return 1 / pval

In [10]:

dfWLO['c2'] = dfWLO.apply(lambda row: c2(row['p1'], row['p2']), axis=1)

NameError: name 'p1' is not defined

In [11]:


def c3(seq):
    mid = len(seq) // 2
    total = sum(seq[mid:])
    M = 0
    copy = seq.copy()
    for i in range(10000):
        random.shuffle(copy)
        tmp = sum(copy[mid:])
        if(tmp <= total): M += 1
    if(M == 0): return math.inf
    else: return 10000/M

In [12]:
dfWLO['c3'] = dfWLO['Sequence'].apply(c3)

In [13]:
def c3(seq):
    mid = len(seq) // 2
    total = sum(seq[mid:])
    M = 0
    copy = seq.copy()
    for i in range(10000):
        random.shuffle(copy)
        tmp = sum(copy[mid:])
        if(tmp <= total): M += 1
    if(M == 0): return math.inf
    else: return 10000/M

In [14]:
dfWLO['c3'] = dfWLO['Sequence'].apply(c3)

In [15]:
dfWLO

,League,Season,Team,Sequence,seq_len,Total_Games,Wins,n2,n1,k2,k1,p1,p2,p,c1,c3
0,Bundesliga,2000,Bayern Munich,"[1, 1, 1, 0, 1, 1, 0, 0, 1, 1, 0, 0, 1, 1, 1, ...",35,29,19,14,15,9,10,0.666667,0.642857,0.655172,1.782253,1.448855
1,Bundesliga,2000,Bochum,"[1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, ...",34,28,7,14,14,5,2,0.142857,0.357143,0.250000,1.125706,5.313496
2,Bundesliga,2000,Cottbus,"[0, 0, 0, 1, 0, 0, 1, 0, 1, 1, 0, 0, 1, 0, 0, ...",35,32,13,16,16,6,7,0.437500,0.375000,0.406250,1.974245,1.307531
3,Bundesliga,2000,Dortmund,"[1, 1, 0, 1, 1, 0, 1, 0, 0, 0, 1, 1, 1, 1, 1, ...",35,25,17,12,13,7,10,0.769231,0.583333,0.680000,3.023294,1.085776
4,Bundesliga,2000,Ein Frankfurt,"[1, 0, 1, 0, 1, 0, 0, 1, 0, 1, 1, 0, 0, 0, 0, ...",34,29,10,14,15,6,4,0.266667,0.428571,0.344828,1.208576,3.307972
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1428,La_Liga,2024,Sociedad,"[0, 1, 0, 0, 0, 1, 1, 0, 1, 1, 0, 1, 1, 0, 1, ...",38,31,13,15,16,8,5,0.312500,0.533333,0.419355,1.141987,5.500550
1429,La_Liga,2024,Valencia,"[0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 1, ...",38,25,11,12,13,2,9,0.692308,0.166667,0.440000,20.544653,1.000901
1430,La_Liga,2024,Valladolid,"[1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, ...",38,34,4,17,17,4,0,0.000000,0.235294,0.117647,1.043195,19.880716
1431,La_Liga,2024,Vallecano,"[1, 0, 0, 1, 1, 0, 1, 0, 0, 0, 1, 1, 1, 1, 1, ...",38,25,13,12,13,6,7,0.538462,0.500000,0.520000,1.793021,1.389468


In [16]:
dfWLO

,League,Season,Team,Sequence,seq_len,Total_Games,Wins,n2,n1,k2,k1,p1,p2,p,c1,c3
0,Bundesliga,2000,Bayern Munich,"[1, 1, 1, 0, 1, 1, 0, 0, 1, 1, 0, 0, 1, 1, 1, ...",35,29,19,14,15,9,10,0.666667,0.642857,0.655172,1.782253,1.448855
1,Bundesliga,2000,Bochum,"[1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, ...",34,28,7,14,14,5,2,0.142857,0.357143,0.250000,1.125706,5.313496
2,Bundesliga,2000,Cottbus,"[0, 0, 0, 1, 0, 0, 1, 0, 1, 1, 0, 0, 1, 0, 0, ...",35,32,13,16,16,6,7,0.437500,0.375000,0.406250,1.974245,1.307531
3,Bundesliga,2000,Dortmund,"[1, 1, 0, 1, 1, 0, 1, 0, 0, 0, 1, 1, 1, 1, 1, ...",35,25,17,12,13,7,10,0.769231,0.583333,0.680000,3.023294,1.085776
4,Bundesliga,2000,Ein Frankfurt,"[1, 0, 1, 0, 1, 0, 0, 1, 0, 1, 1, 0, 0, 0, 0, ...",34,29,10,14,15,6,4,0.266667,0.428571,0.344828,1.208576,3.307972
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1428,La_Liga,2024,Sociedad,"[0, 1, 0, 0, 0, 1, 1, 0, 1, 1, 0, 1, 1, 0, 1, ...",38,31,13,15,16,8,5,0.312500,0.533333,0.419355,1.141987,5.500550
1429,La_Liga,2024,Valencia,"[0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 1, ...",38,25,11,12,13,2,9,0.692308,0.166667,0.440000,20.544653,1.000901
1430,La_Liga,2024,Valladolid,"[1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, ...",38,34,4,17,17,4,0,0.000000,0.235294,0.117647,1.043195,19.880716
1431,La_Liga,2024,Vallecano,"[1, 0, 0, 1, 1, 0, 1, 0, 0, 0, 1, 1, 1, 1, 1, ...",38,25,13,12,13,6,7,0.538462,0.500000,0.520000,1.793021,1.389468
